# Task 4: Evaluate the Baseline Model
**Marks: 2/20** | Pneumonia Detection — Deep Learning Assignment

---

## Overview
In this task we rigorously evaluate the baseline CNN on the **unseen test set** using clinically meaningful metrics.

Training accuracy alone is not enough — especially in healthcare. We need to understand:
- How many pneumonia cases did we **correctly catch**? (Recall)
- How many healthy patients did we **wrongly flag**? (Precision)
- What is the overall **balance** between both? (F1-Score)

## Why These Metrics Matter in Healthcare
| Error Type | Meaning | Consequence |
|------------|---------|-------------|
| False Negative | Missed pneumonia case | Patient goes untreated — potentially fatal |
| False Positive | Healthy patient flagged | Unnecessary tests — less harmful |

This is why **Recall for PNEUMONIA** is our most critical metric.


---
## Step 1: Setup — Import Libraries and Load Model
**What:** Load all required libraries, the saved baseline CNN model, and build the test generator.

**Why:** We saved the model at the end of Task 3 so we don't need to retrain. The test generator loads our 624 unseen test images in the correct format for prediction.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/pneumonia-detection')
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader import build_generators
from src.evaluator import evaluate_model
from src.config import BASELINE_MODEL_PATH
print('Libraries loaded successfully.')

---
## Step 2: Load Saved Model and Test Generator
**What:** Load the baseline CNN we saved in Task 3 and prepare the test generator.

**Why:** We always evaluate on the test set — data the model has **never seen** during training or validation. This gives us an honest measure of real-world performance.

> Important: `shuffle=False` on test generator ensures predictions align with true labels in the correct order.


In [ ]:
baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH)
print('Model loaded:', baseline_model.name)

_, _, test_gen = build_generators()
print('Test generator ready.')
print(f'Test images: {test_gen.samples}')

---
## Step 3: Generate Predictions and Evaluate
**What:** Run the model on all 624 test images and compute accuracy, precision, recall and F1-score.

**Why:** The classification report gives us a complete picture of model performance per class — not just an overall accuracy number which can be misleading with imbalanced datasets.

The `evaluate_model` function:
1. Resets the generator to start from image 1
2. Runs inference on all test images
3. Converts probabilities to binary labels using threshold 0.5
4. Prints full classification report
5. Plots confusion matrix heatmap


In [ ]:
acc, y_true, y_pred, y_prob = evaluate_model(
    baseline_model, test_gen, model_name='Baseline CNN'
)

---
## Step 4: Prediction Confidence Distribution
**What:** Plot histogram of model confidence scores separately for NORMAL and PNEUMONIA images.

**Why:** This tells us how **certain** the model is about its predictions.
- Bars close to 0 → model strongly predicts NORMAL
- Bars close to 1 → model strongly predicts PNEUMONIA
- Bars near 0.5 → model is uncertain — these are the hard cases

A well-calibrated model should show most predictions far from the 0.5 boundary.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Baseline CNN — Prediction Confidence Distribution',
             fontsize=13, fontweight='bold')

for ax, cls, val in zip(axes, ['NORMAL', 'PNEUMONIA'], [0, 1]):
    mask  = y_true == val
    probs = y_prob[mask]
    ax.hist(probs, bins=30,
            color='#4C72B0' if val == 0 else '#DD8452',
            edgecolor='white', alpha=0.85)
    ax.axvline(0.5, color='red', linestyle='--', label='Decision threshold (0.5)')
    ax.set_title(f'True {cls} images')
    ax.set_xlabel('Predicted probability (PNEUMONIA)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Observations and Analysis

**1. What do the 4 boxes in the confusion matrix represent?**
- True Positive (TP): Correctly predicted PNEUMONIA — 349 (bottom-right)
- True Negative (TN): Correctly predicted NORMAL — 202 (top-left)
- False Positive (FP): Predicted PNEUMONIA but actually NORMAL — 32 (top-right) — Type I error
- False Negative (FN): Predicted NORMAL but actually PNEUMONIA — 41 (bottom-left) — Type II error — most dangerous in healthcare!

**2. Which metric matters most and why?**
Recall (Sensitivity) for PNEUMONIA is the most critical metric. A False Negative means missing a real pneumonia case — leading to delayed treatment and potentially fatal outcomes. False Positives only cause unnecessary follow-up tests, which is far less harmful. Therefore minimizing False Negatives (maximizing Recall) is the priority in any medical screening tool.

**3. Is 89% recall good enough for a real hospital system?**
No — 89% recall means missing 11% of pneumonia cases. Out of 390 actual pneumonia cases in the test set, 41 were missed. In a real clinical setting this is medically unacceptable. Real screening tools typically aim for recall above 95%, sometimes sacrificing precision (accepting more false alarms) to ensure minimal missed cases. The exact acceptable threshold would be determined by medical experts and regulatory bodies.

**4. What would improve these numbers?**
- Address class imbalance via weighted loss functions, SMOTE oversampling, or undersampling
- Advanced augmentation: elastic transformations, color jittering, cutout
- Transfer Learning with pretrained models like ResNet, EfficientNet, MobileNetV2 (Task 5)
- Hyperparameter tuning: learning rate, dropout rate, batch size
- Ensemble methods: combine predictions from multiple models
- Lower classification threshold below 0.5 to increase recall at cost of precision
- Larger validation set — current 16 images is too small for reliable evaluation

---
## Key Takeaway
> The baseline CNN achieves 88% accuracy and 89% pneumonia recall. While promising, the 41 missed pneumonia cases highlight the need for a stronger model. Task 5 addresses this using Transfer Learning with MobileNetV2.
